# Distributed Quadratic Form Training

Train a quadratic model `<circuit | mps_h · mx · mps | circuit>` using `EngineDistributed`.

- **Loss**: NLL `-mean(log(P) + log_scale)`
- **Optimizer**: SGDG
- **Distributed**: uses `torch.distributed` (gloo backend), falls back to single-process

> **Note**: Distributed training via `torchrun` cannot run inside a notebook.
> This notebook runs in **single-process mode** for demonstration.
> For multi-process distributed training, use:
> ```bash
> torchrun --nproc_per_node=2 examples/train_dist.py
> ```

In [ ]:
import os
os.chdir(os.path.join(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()))

%matplotlib inline
import torch
import numpy as np
import matplotlib.pyplot as plt

from tneq_qc import (
    QCTN, BackendFactory, Quadratic,
    DataGenerator, make_data_fn, SGDG,
)
from tneq_qc.distributed import EngineDistributed
from tneq_qc.distributed.engine.distributed_engine import PartitionConfig

## Configuration

In [ ]:
N_QUBITS   = 4
BOND_DIM   = 2
PHYS_DIM   = 2
BATCH_SIZE = 128
N_STEPS    = 200
LR         = 0.01
LOG_EVERY  = 10
DEVICE     = "cpu"  # change to "cuda" for GPU

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
def init_circuit_01(qctn, backend):
    """Fill each circuit core with an alternating 0/1 pattern."""
    for c in qctn.cores:
        core = qctn.cores_weights[c]
        shape = tuple(core.shape)
        n = 1
        for d in shape:
            n *= d
        flat = torch.zeros(n, dtype=core.dtype)
        for i in range(n):
            flat[i] = float(i % 2)
        qctn.cores_weights[c] = backend.convert_to_tensor(flat.reshape(shape))
    return qctn

## Build Model & Engine

In notebook mode, we run single-process (`rank=0`, `world_size=1`).
`EngineDistributed` works transparently in this case.

In [ ]:
rank = 0
world_size = 1

backend = BackendFactory.create_backend('pytorch', device=DEVICE, dtype='float32')
data_gen = DataGenerator(backend, mx_K=PHYS_DIM)

# Build quadratic model
model = Quadratic(nqubits=N_QUBITS, bond_dim=BOND_DIM, phys_dim=PHYS_DIM,
                  backend=backend).auto_init()
init_circuit_01(model._submodules['circuit'], backend)
model._submodules['mps'].requires_grad_(True)
combined = model.build()

print(f"Combined: {combined.ncores} cores, {len(combined.parameters())} trainable")

# Create distributed engine
from tneq_qc.distributed.comm import get_comm_backend

comm = get_comm_backend(backend='auto', rank=rank, world_size=world_size)
engine = EngineDistributed(
    backend=backend,
    strategy_mode='full',
    comm=comm,
    partition_config=PartitionConfig(strategy='layer', num_partitions=world_size),
)
engine.init_distributed(combined)
print(f"Engine ready (world_size={world_size})")

## Training

In [ ]:
optimizer = SGDG(combined.parameters(), backend, lr=LR / world_size)
data_fn = make_data_fn(data_gen, combined, batch_size=BATCH_SIZE, K=PHYS_DIM)
loss_history = []

for step in range(1, N_STEPS + 1):
    data_fn(step)
    loss_val, grads = engine.contract_for_gradient(combined, target=1, loss='nll')
    optimizer.step(list(grads))
    lv = float(loss_val)
    loss_history.append(lv)
    if step % LOG_EVERY == 0 or step == 1:
        print(f"  Step {step:4d}/{N_STEPS}  loss={lv:.6f}")

print(f"\nDone. Initial={loss_history[0]:.6f}  Final={loss_history[-1]:.6f}")

## Save Model

In [ ]:
save_dir = "checkpoints"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "dist_mps.safetensors")
model._submodules['mps'].save_cores(save_path, metadata={
    'n_qubits': str(N_QUBITS),
    'bond_dim': str(BOND_DIM),
    'n_steps': str(N_STEPS),
    'final_loss': f"{loss_history[-1]:.6f}",
})
print(f"Model saved: {save_path}")

## Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_history)
ax.set_xlabel('Step')
ax.set_ylabel('NLL Loss')
ax.set_title(f'Distributed Training Loss ({N_QUBITS} qubits, world_size={world_size})')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Validation: Reload & Verify

In [ ]:
from tneq_qc.core.tn_tensor import TNTensor

# Rebuild model and load saved weights
model_val = Quadratic(nqubits=N_QUBITS, bond_dim=BOND_DIM, phys_dim=PHYS_DIM,
                      backend=backend).auto_init()
init_circuit_01(model_val._submodules['circuit'], backend)
model_val._submodules['mps'].load_cores(save_path)
combined_val = model_val.build()

# Compare core values
mps_orig = model._submodules['mps']
mps_loaded = model_val._submodules['mps']

print("Per-core comparison (trained vs loaded):")
for core_name in mps_orig.cores:
    orig = mps_orig.cores_weights[core_name]
    load = mps_loaded.cores_weights[core_name]
    orig_np = backend.tensor_to_numpy(orig.tensor * orig.scale if isinstance(orig, TNTensor) else orig)
    load_np = backend.tensor_to_numpy(load.tensor * load.scale if isinstance(load, TNTensor) else load)
    err = np.max(np.abs(orig_np - load_np))
    print(f"  Core '{core_name}': max_abs_error={err:.2e}")